# 05_04 - Model Comparison: Baselines vs Quantile Forecasts
## 1. Methodology Overview

This notebook compares the actual baseline suite against the quantile suite using the project's own evaluation code.
The source implementation is in `src/models/train_model.py` and `src/models/evaluate_model.py`.

**Source data used in this notebook:**
- `data/processed/train_features.csv`
- `data/processed/validation_features.csv`

The comparison is grounded in the repository's real processed data and follows the same logic used by the pipeline.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = Path('../../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

train_df = pd.read_csv(Path('../../data/processed/train_features.csv'))
validation_df = pd.read_csv(Path('../../data/processed/validation_features.csv'))

print(f'Train features: {train_df.shape[0]} rows x {train_df.shape[1]} columns')
print(f'Validation features: {validation_df.shape[0]} rows x {validation_df.shape[1]} columns')
display(train_df.head())

Train features: 1461 rows x 155 columns
Validation features: 366 rows x 155 columns


,Date,Spot_Price_SPEL,Future_M1_Price,Future_M1_OpenInterest,Future_M2_Price,Future_M2_OpenInterest,Future_M3_Price,Future_M3_OpenInterest,Future_M4_Price,Future_M4_OpenInterest,...,future_m5_oi_change_1d,future_m5_oi_change_7d,future_m5_oi_pct_change_1d,future_m5_oi_pct_change_7d,future_m6_oi_change_1d,future_m6_oi_change_7d,future_m6_oi_pct_change_1d,future_m6_oi_pct_change_7d,front_month_premium,front_month_premium_rel
0,2020-01-01,35.54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-02,40.00,44.81,1383.0,41.7,1397.0,38.54,10.0,42.11,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.81,0.120250
2,2020-01-03,39.51,45.25,1383.0,41.9,1405.0,38.90,10.0,42.50,0.0,...,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,5.74,0.145280
3,2020-01-04,35.67,45.25,1383.0,41.9,1405.0,38.90,10.0,42.50,0.0,...,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,9.58,0.268573
4,2020-01-05,38.18,45.25,1383.0,41.9,1405.0,38.90,10.0,42.50,0.0,...,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,7.07,0.185175


## 2. Train Both Model Families

The baseline family gives interpretable reference points, while the quantile family provides uncertainty-aware forecasts.
Both are trained on the same project splits so the comparison is fair and fully reproducible.

In [2]:
from src.models.train_model import train_baseline_suite, train_quantile_suite

baseline_output = train_baseline_suite(train_df=train_df, eval_df=validation_df)
quantile_output = train_quantile_suite(train_df=train_df, eval_df=validation_df)

display(baseline_output.summary)
display(quantile_output.summary)


,model_name,mae,rmse
0,naive_last_value,14.993534,21.187930
1,linear_regression_baseline,19.098751,24.073276
2,rolling_mean_7,21.366524,27.617335
3,seasonal_naive_lag_7,26.656089,34.927369


,model_name,quantile,pinball_loss,mae,rmse
0,gbr_quantile_0.5,0.50,7.977632,15.955264,20.194186
1,gbr_quantile_0.9,0.90,4.857351,22.790294,28.525883
2,gbr_quantile_0.95,0.95,3.840220,26.262822,32.755788


## 3. Unified Comparison Table

The comparison helper in `src/models/evaluate_model.py` merges the two model families into a single table so that RMSE, MAE, and quantile pinball loss can be inspected side by side.

In [5]:
from src.models.evaluate_model import compare_all_models

comparison_df = compare_all_models(baseline_output.results, quantile_output.results)
comparison_df = comparison_df.sort_values(['model_type', 'rmse', 'mae']).reset_index(drop=True)
display(comparison_df)


best_row = comparison_df.sort_values(['rmse', 'mae']).iloc[0]

print(best_row)
print("----------------")

print(f"Best model in this comparison: {best_row['model_name']}")
print(f"Best RMSE: {best_row['rmse']:.4f}")

,model_type,model_name,quantile,mae,rmse,pinball_loss
0,baseline,naive_last_value,<NA>,14.993534,21.187930,<NA>
1,baseline,linear_regression_baseline,<NA>,19.098751,24.073276,<NA>
2,baseline,rolling_mean_7,<NA>,21.366524,27.617335,<NA>
3,baseline,seasonal_naive_lag_7,<NA>,26.656089,34.927369,<NA>
4,quantile,gbr_quantile_0.5,0.5,15.955264,20.194186,7.977632
5,quantile,gbr_quantile_0.9,0.9,22.790294,28.525883,4.857351
6,quantile,gbr_quantile_0.95,0.95,26.262822,32.755788,3.84022


model_type              quantile
model_name      gbr_quantile_0.5
quantile                     0.5
mae                    15.955264
rmse                   20.194186
pinball_loss            7.977632
Name: 4, dtype: object
----------------
Best model in this comparison: gbr_quantile_0.5
Best RMSE: 20.1942


## 4. Handoff

This notebook closes the modeling stage by showing which model family is more useful for the downstream policy layer.
The next stage uses the quantile outputs as policy inputs and moves into decision-making and backtesting.